# 01 - Autoencoders and Variational Autoencoders (VAEs) [OPTIONAL]

---

> **This notebook is OPTIONAL.** It covers generative modeling with autoencoders and VAEs.
> These topics are not required for the core curriculum but provide valuable insight into
> unsupervised representation learning and generative models.

---

## Learning Objectives

By the end of this notebook, you will be able to:

- Explain the **encoder-decoder** architecture and the concept of a **latent space**
- Build and train a standard **autoencoder** for image reconstruction
- Understand the **reparameterization trick** and why it is needed
- Derive and implement the **VAE loss** (reconstruction + KL divergence)
- Visualize the latent space of a VAE and **generate new samples**

## Prerequisites

- **DL100**: Neural network fundamentals (forward pass, backpropagation)
- **DL150**: PyTorch foundations (`nn.Module`, optimizers, tensors)
- **DL200**: MLP training (training loops, loss functions)
- Basic probability: Gaussian distributions, KL divergence (intuition)

## Table of Contents

1. [Setup and Imports](#1-setup-and-imports)
2. [What is an Autoencoder?](#2-what-is-an-autoencoder)
3. [Build a Standard Autoencoder](#3-build-a-standard-autoencoder)
4. [Train the Autoencoder on MNIST](#4-train-the-autoencoder-on-mnist)
5. [Visualize Reconstructions](#5-visualize-reconstructions)
6. [From Autoencoder to VAE](#6-from-autoencoder-to-vae)
7. [The Reparameterization Trick](#7-the-reparameterization-trick)
8. [Build and Train a VAE](#8-build-and-train-a-vae)
9. [Visualize the Latent Space](#9-visualize-the-latent-space)
10. [Generate New Samples](#10-generate-new-samples)
11. [Exercise: Bottleneck Size Experiment](#11-exercise-bottleneck-size-experiment)
12. [Common Mistakes & Debugging Tips](#12-common-mistakes--debugging-tips)

---

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed
from src.utils.device import get_device
from src.utils.plotting import plot_loss_curves

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

set_global_seed(42)
device = get_device()

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

print("Setup complete.")

In [ ]:
# Load MNIST dataset (fallback to sklearn digits if torchvision unavailable)
try:
    from torchvision import datasets, transforms

    transform = transforms.Compose([transforms.ToTensor()])
    train_dataset = datasets.MNIST(root="../../data", train=True, download=True, transform=transform)
    test_dataset = datasets.MNIST(root="../../data", train=False, download=True, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

    INPUT_DIM = 784  # 28x28
    IMG_SHAPE = (1, 28, 28)
    USE_TORCHVISION = True
    print(f"MNIST loaded: {len(train_dataset)} train, {len(test_dataset)} test")

except ImportError:
    from sklearn.datasets import load_digits
    from sklearn.model_selection import train_test_split

    digits = load_digits()
    X = digits.data / 16.0  # normalize to [0, 1]
    y = digits.target

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    train_ds = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32).unsqueeze(1).view(-1, 1, 8, 8),
        torch.tensor(y_train, dtype=torch.long),
    )
    test_ds = TensorDataset(
        torch.tensor(X_test, dtype=torch.float32).unsqueeze(1).view(-1, 1, 8, 8),
        torch.tensor(y_test, dtype=torch.long),
    )

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

    INPUT_DIM = 64  # 8x8
    IMG_SHAPE = (1, 8, 8)
    USE_TORCHVISION = False
    print(f"Sklearn digits loaded (torchvision not available): {len(train_ds)} train, {len(test_ds)} test")

---

## 2. What is an Autoencoder?

An **autoencoder** is a neural network that learns to **compress** input data into a low-dimensional representation (the **latent space** or **bottleneck**) and then **reconstruct** the original input from that representation.

### Architecture

```
Input (x) --> [Encoder] --> Bottleneck (z) --> [Decoder] --> Reconstruction (x_hat)
  784-d         ...           d_latent           ...            784-d
```

- **Encoder** $f_\phi$: maps input $\mathbf{x}$ to latent code $\mathbf{z} = f_\phi(\mathbf{x})$
- **Bottleneck** (latent space): low-dimensional representation $\mathbf{z} \in \mathbb{R}^{d}$ with $d \ll D$
- **Decoder** $g_\theta$: reconstructs from latent code $\hat{\mathbf{x}} = g_\theta(\mathbf{z})$

### Loss Function

The autoencoder minimizes **reconstruction loss**:

$$\mathcal{L}_{\text{recon}} = \frac{1}{N}\sum_{i=1}^{N} \|\mathbf{x}_i - \hat{\mathbf{x}}_i\|^2 \quad \text{(MSE)}$$

or, for data normalized to $[0, 1]$:

$$\mathcal{L}_{\text{recon}} = -\frac{1}{N}\sum_{i=1}^{N}\left[\mathbf{x}_i \log \hat{\mathbf{x}}_i + (1-\mathbf{x}_i)\log(1-\hat{\mathbf{x}}_i)\right] \quad \text{(BCE)}$$

### Key Insight

By forcing data through a bottleneck, the network must learn a **compact, meaningful representation** -- it cannot simply memorize (if the bottleneck is small enough).

---

## 3. Build a Standard Autoencoder

In [ ]:
class Autoencoder(nn.Module):
    """Standard autoencoder with fully-connected layers."""

    def __init__(self, input_dim=784, latent_dim=32):
        super().__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim

        # Encoder: input_dim -> 256 -> 128 -> latent_dim
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim),
        )

        # Decoder: latent_dim -> 128 -> 256 -> input_dim
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, input_dim),
            nn.Sigmoid(),  # output in [0, 1] to match pixel range
        )

    def encode(self, x):
        """Encode input to latent representation."""
        return self.encoder(x)

    def decode(self, z):
        """Decode latent representation to reconstruction."""
        return self.decoder(z)

    def forward(self, x):
        """Full forward pass: encode then decode."""
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z


# Instantiate
ae_model = Autoencoder(input_dim=INPUT_DIM, latent_dim=32).to(device)
print(ae_model)
n_params = sum(p.numel() for p in ae_model.parameters())
print(f"\nTotal parameters: {n_params:,}")

---

## 4. Train the Autoencoder on MNIST

In [ ]:
def train_autoencoder(model, train_loader, test_loader, epochs=20, lr=1e-3, device=device):
    """Train a standard autoencoder with MSE reconstruction loss."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(1, epochs + 1):
        # Training
        model.train()
        train_losses = []
        for batch in train_loader:
            if USE_TORCHVISION:
                x, _ = batch
                x = x.view(x.size(0), -1).to(device)  # flatten
            else:
                x, _ = batch
                x = x.view(x.size(0), -1).to(device)

            optimizer.zero_grad()
            x_hat, z = model(x)
            loss = loss_fn(x_hat, x)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        # Validation
        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in test_loader:
                if USE_TORCHVISION:
                    x, _ = batch
                    x = x.view(x.size(0), -1).to(device)
                else:
                    x, _ = batch
                    x = x.view(x.size(0), -1).to(device)
                x_hat, z = model(x)
                val_losses.append(loss_fn(x_hat, x).item())

        avg_train = np.mean(train_losses)
        avg_val = np.mean(val_losses)
        history["train_loss"].append(avg_train)
        history["val_loss"].append(avg_val)

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{epochs} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")

    return history

In [ ]:
set_global_seed(42)
ae_model = Autoencoder(input_dim=INPUT_DIM, latent_dim=32).to(device)
ae_history = train_autoencoder(ae_model, train_loader, test_loader, epochs=20, lr=1e-3)

In [ ]:
plot_loss_curves(ae_history, title="Autoencoder Training Curves")

---

## 5. Visualize Reconstructions

In [ ]:
def visualize_reconstructions(model, data_loader, n=10, img_shape=IMG_SHAPE, device=device):
    """Show original images and their reconstructions side by side."""
    model.eval()
    batch = next(iter(data_loader))
    x, _ = batch
    x_flat = x.view(x.size(0), -1).to(device)

    with torch.no_grad():
        x_hat, _ = model(x_flat)

    x_np = x_flat[:n].cpu().numpy()
    x_hat_np = x_hat[:n].cpu().numpy()

    h, w = img_shape[1], img_shape[2]

    fig, axes = plt.subplots(2, n, figsize=(n * 1.5, 3))
    for i in range(n):
        # Original
        axes[0, i].imshow(x_np[i].reshape(h, w), cmap="gray")
        axes[0, i].axis("off")
        if i == 0:
            axes[0, i].set_title("Original", fontsize=10)

        # Reconstruction
        axes[1, i].imshow(x_hat_np[i].reshape(h, w), cmap="gray")
        axes[1, i].axis("off")
        if i == 0:
            axes[1, i].set_title("Reconstructed", fontsize=10)

    plt.suptitle("Autoencoder Reconstructions", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()


visualize_reconstructions(ae_model, test_loader)

---

## 6. From Autoencoder to VAE

A standard autoencoder learns a **deterministic** mapping $\mathbf{x} \to \mathbf{z}$. The latent space has no structure -- nearby points in latent space may decode to very different outputs, and there are "holes" where decoding produces garbage.

A **Variational Autoencoder (VAE)** fixes this by making the encoder output a **probability distribution** rather than a fixed point:

### VAE Encoder

Instead of $\mathbf{z} = f_\phi(\mathbf{x})$, the encoder outputs:

$$\boldsymbol{\mu} = f_\mu(\mathbf{x}), \quad \log \boldsymbol{\sigma}^2 = f_{\sigma}(\mathbf{x})$$

Then $\mathbf{z}$ is **sampled** from $q(\mathbf{z}|\mathbf{x}) = \mathcal{N}(\boldsymbol{\mu}, \text{diag}(\boldsymbol{\sigma}^2))$.

### VAE Loss

The VAE loss has two terms:

$$\mathcal{L}_{\text{VAE}} = \underbrace{\mathcal{L}_{\text{recon}}}_{\text{reconstruction}} + \underbrace{D_{KL}\left(q(\mathbf{z}|\mathbf{x}) \,\|\, p(\mathbf{z})\right)}_{\text{KL divergence}}$$

where the KL divergence term has a closed-form solution for Gaussians:

$$D_{KL} = -\frac{1}{2} \sum_{j=1}^{d} \left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

- **Reconstruction term**: ensures the decoder can recover the input
- **KL term**: regularizes the latent space to be close to a standard normal $\mathcal{N}(0, I)$, creating a smooth, continuous latent space

---

## 7. The Reparameterization Trick

**Problem:** We need to sample $\mathbf{z} \sim \mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\sigma}^2)$, but sampling is a non-differentiable operation. We cannot backpropagate through a random sample.

**Solution -- the reparameterization trick:**

Instead of sampling $\mathbf{z}$ directly, we:

1. Sample $\boldsymbol{\epsilon} \sim \mathcal{N}(0, \mathbf{I})$ (this does not depend on model parameters)
2. Compute $\mathbf{z} = \boldsymbol{\mu} + \boldsymbol{\sigma} \cdot \boldsymbol{\epsilon}$

Now the randomness is "externalized" to $\boldsymbol{\epsilon}$, and gradients can flow through $\boldsymbol{\mu}$ and $\boldsymbol{\sigma}$.

$$\mathbf{z} = \boldsymbol{\mu} + \boldsymbol{\sigma} \odot \boldsymbol{\epsilon}, \quad \boldsymbol{\epsilon} \sim \mathcal{N}(0, \mathbf{I})$$

In [ ]:
# Illustrate the reparameterization trick
set_global_seed(42)

mu = torch.tensor([1.0, -0.5], requires_grad=True)
log_var = torch.tensor([0.2, -0.3], requires_grad=True)
sigma = torch.exp(0.5 * log_var)

# Reparameterization
epsilon = torch.randn_like(mu)  # sample from N(0, I)
z = mu + sigma * epsilon         # shift and scale

print(f"mu:      {mu.data}")
print(f"log_var: {log_var.data}")
print(f"sigma:   {sigma.data}")
print(f"epsilon: {epsilon.data}")
print(f"z:       {z.data}")

# Verify gradients flow
loss = z.sum()
loss.backward()
print(f"\ngrad(mu):      {mu.grad}  (should be [1, 1])")
print(f"grad(log_var): {log_var.grad}  (should be non-zero)")

---

## 8. Build and Train a VAE

In [ ]:
class VAE(nn.Module):
    """Variational Autoencoder with fully-connected layers."""

    def __init__(self, input_dim=784, latent_dim=2):
        super().__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim

        # Encoder: shared layers
        self.encoder_shared = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
        )
        # Encoder: separate heads for mu and log_var
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_log_var = nn.Linear(128, latent_dim)

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, input_dim),
            nn.Sigmoid(),
        )

    def encode(self, x):
        """Encode input to latent distribution parameters."""
        h = self.encoder_shared(x)
        mu = self.fc_mu(h)
        log_var = self.fc_log_var(h)
        return mu, log_var

    def reparameterize(self, mu, log_var):
        """Sample z using the reparameterization trick."""
        sigma = torch.exp(0.5 * log_var)
        epsilon = torch.randn_like(sigma)
        z = mu + sigma * epsilon
        return z

    def decode(self, z):
        """Decode latent vector to reconstruction."""
        return self.decoder(z)

    def forward(self, x):
        """Full forward pass."""
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        x_hat = self.decode(z)
        return x_hat, mu, log_var, z


# Instantiate with latent_dim=2 for easy visualization
vae_model = VAE(input_dim=INPUT_DIM, latent_dim=2).to(device)
print(vae_model)
n_params = sum(p.numel() for p in vae_model.parameters())
print(f"\nTotal parameters: {n_params:,}")

In [ ]:
def vae_loss_fn(x, x_hat, mu, log_var):
    """Compute VAE loss = Reconstruction (BCE) + KL Divergence.

    Args:
        x: Original input, shape (batch, input_dim)
        x_hat: Reconstruction, shape (batch, input_dim)
        mu: Mean of latent distribution, shape (batch, latent_dim)
        log_var: Log-variance of latent distribution, shape (batch, latent_dim)

    Returns:
        total_loss, recon_loss, kl_loss (all scalars)
    """
    # Reconstruction loss: binary cross-entropy (sum over features, mean over batch)
    recon_loss = F.binary_cross_entropy(x_hat, x, reduction='sum') / x.size(0)

    # KL divergence: -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / x.size(0)

    total_loss = recon_loss + kl_loss
    return total_loss, recon_loss, kl_loss

In [ ]:
def train_vae(model, train_loader, test_loader, epochs=30, lr=1e-3, device=device):
    """Train a VAE and return loss history."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": [], "train_recon": [], "train_kl": []}

    for epoch in range(1, epochs + 1):
        # Training
        model.train()
        t_total, t_recon, t_kl = [], [], []
        for batch in train_loader:
            x, _ = batch
            x = x.view(x.size(0), -1).to(device)

            optimizer.zero_grad()
            x_hat, mu, log_var, z = model(x)
            loss, recon, kl = vae_loss_fn(x, x_hat, mu, log_var)
            loss.backward()
            optimizer.step()

            t_total.append(loss.item())
            t_recon.append(recon.item())
            t_kl.append(kl.item())

        # Validation
        model.eval()
        v_total = []
        with torch.no_grad():
            for batch in test_loader:
                x, _ = batch
                x = x.view(x.size(0), -1).to(device)
                x_hat, mu, log_var, z = model(x)
                loss, _, _ = vae_loss_fn(x, x_hat, mu, log_var)
                v_total.append(loss.item())

        avg_total = np.mean(t_total)
        avg_recon = np.mean(t_recon)
        avg_kl = np.mean(t_kl)
        avg_val = np.mean(v_total)

        history["train_loss"].append(avg_total)
        history["val_loss"].append(avg_val)
        history["train_recon"].append(avg_recon)
        history["train_kl"].append(avg_kl)

        if epoch % 5 == 0 or epoch == 1:
            print(
                f"Epoch {epoch:3d}/{epochs} | Total: {avg_total:.2f} | "
                f"Recon: {avg_recon:.2f} | KL: {avg_kl:.2f} | Val: {avg_val:.2f}"
            )

    return history

In [ ]:
set_global_seed(42)
vae_model = VAE(input_dim=INPUT_DIM, latent_dim=2).to(device)
vae_history = train_vae(vae_model, train_loader, test_loader, epochs=30, lr=1e-3)

In [ ]:
# Plot total loss and the two components
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs_range = range(1, len(vae_history["train_loss"]) + 1)

axes[0].plot(epochs_range, vae_history["train_loss"], label="Train")
axes[0].plot(epochs_range, vae_history["val_loss"], label="Val")
axes[0].set_title("Total VAE Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, vae_history["train_recon"], color="green")
axes[1].set_title("Reconstruction Loss (BCE)")
axes[1].set_xlabel("Epoch")
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_range, vae_history["train_kl"], color="red")
axes[2].set_title("KL Divergence")
axes[2].set_xlabel("Epoch")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# VAE reconstructions
def visualize_vae_reconstructions(model, data_loader, n=10, img_shape=IMG_SHAPE, device=device):
    """Show original images and VAE reconstructions."""
    model.eval()
    batch = next(iter(data_loader))
    x, _ = batch
    x_flat = x.view(x.size(0), -1).to(device)

    with torch.no_grad():
        x_hat, mu, log_var, z = model(x_flat)

    x_np = x_flat[:n].cpu().numpy()
    x_hat_np = x_hat[:n].cpu().numpy()
    h, w = img_shape[1], img_shape[2]

    fig, axes = plt.subplots(2, n, figsize=(n * 1.5, 3))
    for i in range(n):
        axes[0, i].imshow(x_np[i].reshape(h, w), cmap="gray")
        axes[0, i].axis("off")
        if i == 0:
            axes[0, i].set_title("Original", fontsize=10)
        axes[1, i].imshow(x_hat_np[i].reshape(h, w), cmap="gray")
        axes[1, i].axis("off")
        if i == 0:
            axes[1, i].set_title("VAE Recon", fontsize=10)

    plt.suptitle("VAE Reconstructions (latent_dim=2)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()


visualize_vae_reconstructions(vae_model, test_loader)

---

## 9. Visualize the Latent Space

Because our VAE uses `latent_dim=2`, we can directly plot the latent codes as a 2D scatter plot, colored by digit class.

In [ ]:
@torch.no_grad()
def plot_latent_space(model, data_loader, device=device):
    """Scatter plot of 2D latent codes colored by digit class."""
    model.eval()
    all_mu = []
    all_labels = []

    for batch in data_loader:
        x, y = batch
        x = x.view(x.size(0), -1).to(device)
        mu, log_var = model.encode(x)
        all_mu.append(mu.cpu().numpy())
        all_labels.append(y.numpy())

    all_mu = np.concatenate(all_mu, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    fig, ax = plt.subplots(figsize=(8, 6))
    scatter = ax.scatter(
        all_mu[:, 0], all_mu[:, 1],
        c=all_labels, cmap="tab10", s=2, alpha=0.5
    )
    cbar = plt.colorbar(scatter, ax=ax, ticks=range(10))
    cbar.set_label("Digit Class")
    ax.set_xlabel("$z_1$")
    ax.set_ylabel("$z_2$")
    ax.set_title("VAE Latent Space (2D)")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_latent_space(vae_model, test_loader)

**Observations:**
- Different digit classes form clusters in latent space
- The KL divergence regularizer keeps the distribution roughly Gaussian
- Nearby points in latent space correspond to similar-looking digits

---

## 10. Generate New Samples

Since the VAE latent space is regularized to $\mathcal{N}(0, I)$, we can **generate new images** by sampling from a standard normal and decoding.

In [ ]:
@torch.no_grad()
def generate_samples(model, n_samples=20, img_shape=IMG_SHAPE, device=device):
    """Generate new images by sampling from the latent space."""
    model.eval()
    # Sample from standard normal
    z = torch.randn(n_samples, model.latent_dim).to(device)
    samples = model.decode(z).cpu().numpy()

    h, w = img_shape[1], img_shape[2]
    cols = min(n_samples, 10)
    rows = (n_samples + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.5))
    if rows == 1:
        axes = axes.reshape(1, -1)

    for i in range(n_samples):
        r, c = divmod(i, cols)
        axes[r, c].imshow(samples[i].reshape(h, w), cmap="gray")
        axes[r, c].axis("off")

    # Hide unused axes
    for i in range(n_samples, rows * cols):
        r, c = divmod(i, cols)
        axes[r, c].axis("off")

    plt.suptitle("Generated Samples (random z ~ N(0, I))", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()


set_global_seed(42)
generate_samples(vae_model, n_samples=20)

In [ ]:
@torch.no_grad()
def plot_latent_grid(model, n=15, z_range=(-3, 3), img_shape=IMG_SHAPE, device=device):
    """Decode a grid of points in 2D latent space to visualize the manifold."""
    model.eval()
    h, w = img_shape[1], img_shape[2]

    # Create grid of z values
    z1 = np.linspace(z_range[0], z_range[1], n)
    z2 = np.linspace(z_range[0], z_range[1], n)

    canvas = np.zeros((n * h, n * w))

    for i, z2_val in enumerate(reversed(z2)):  # reversed so y-axis goes up
        for j, z1_val in enumerate(z1):
            z = torch.tensor([[z1_val, z2_val]], dtype=torch.float32).to(device)
            sample = model.decode(z).cpu().numpy().reshape(h, w)
            canvas[i * h:(i + 1) * h, j * w:(j + 1) * w] = sample

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(canvas, cmap="gray")
    ax.set_title(f"Latent Space Manifold ({n}x{n} grid, z in [{z_range[0]}, {z_range[1]}])")
    ax.axis("off")
    plt.tight_layout()
    plt.show()


plot_latent_grid(vae_model, n=15, z_range=(-3, 3))

**Key observations:**
- The grid shows smooth transitions between digit classes
- The VAE learns a continuous manifold -- interpolating between points produces plausible images
- This is the key advantage over standard autoencoders

---

## 11. Exercise: Bottleneck Size Experiment

**Task:** Modify the latent dimension (bottleneck size) and observe the effect on reconstruction quality.

1. Train VAEs with `latent_dim` values of `2`, `8`, `16`, and `32`
2. Compare reconstruction quality visually
3. Compare final validation loss

**Questions to answer:**
- How does increasing the bottleneck size affect reconstruction quality?
- How does it affect the KL divergence?
- What is the trade-off?

```python
# Your code here
latent_dims = [2, 8, 16, 32]

for ld in latent_dims:
    set_global_seed(42)
    model = VAE(input_dim=INPUT_DIM, latent_dim=ld).to(device)
    # TODO: train for 15 epochs, print final losses
    # TODO: visualize reconstructions for each
    pass
```

In [ ]:
# Try the exercise yourself before looking at the solution!






### Solution

In [ ]:
# ----- Solution: Bottleneck Size Experiment -----

latent_dims = [2, 8, 16, 32]
results = {}

for ld in latent_dims:
    print(f"\n{'='*50}")
    print(f"Training VAE with latent_dim = {ld}")
    print(f"{'='*50}")

    set_global_seed(42)
    model = VAE(input_dim=INPUT_DIM, latent_dim=ld).to(device)
    history = train_vae(model, train_loader, test_loader, epochs=15, lr=1e-3)
    results[ld] = {"model": model, "history": history}

In [ ]:
# Compare validation losses
fig, ax = plt.subplots(figsize=(8, 5))
for ld, res in results.items():
    final_val = res["history"]["val_loss"][-1]
    ax.plot(res["history"]["val_loss"], label=f"dim={ld} (final={final_val:.1f})")

ax.set_xlabel("Epoch")
ax.set_ylabel("Validation Loss")
ax.set_title("VAE Validation Loss vs Latent Dimension")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nSummary:")
print(f"{'Latent Dim':>10} | {'Final Val Loss':>14} | {'Final Recon':>11} | {'Final KL':>8}")
print("-" * 55)
for ld, res in results.items():
    h = res["history"]
    print(f"{ld:>10d} | {h['val_loss'][-1]:>14.2f} | {h['train_recon'][-1]:>11.2f} | {h['train_kl'][-1]:>8.2f}")

In [ ]:
# Visual comparison of reconstructions
model_eval = results[2]["model"]
model_eval.eval()

# Get a fixed batch for comparison
batch = next(iter(test_loader))
x, _ = batch
x_flat = x.view(x.size(0), -1).to(device)
n_show = 8
h, w = IMG_SHAPE[1], IMG_SHAPE[2]

fig, axes = plt.subplots(len(latent_dims) + 1, n_show, figsize=(n_show * 1.5, (len(latent_dims) + 1) * 1.5))

# Top row: originals
for i in range(n_show):
    axes[0, i].imshow(x_flat[i].cpu().numpy().reshape(h, w), cmap="gray")
    axes[0, i].axis("off")
axes[0, 0].set_ylabel("Original", fontsize=10, rotation=0, labelpad=60)

# Remaining rows: reconstructions for each latent dim
for row, ld in enumerate(latent_dims, start=1):
    model_ld = results[ld]["model"]
    model_ld.eval()
    with torch.no_grad():
        x_hat, _, _, _ = model_ld(x_flat)
    for i in range(n_show):
        axes[row, i].imshow(x_hat[i].cpu().numpy().reshape(h, w), cmap="gray")
        axes[row, i].axis("off")
    axes[row, 0].set_ylabel(f"dim={ld}", fontsize=10, rotation=0, labelpad=50)

plt.suptitle("Reconstruction Quality vs Latent Dimension", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nKey observations:")
print("- Larger latent dim -> better reconstruction (lower recon loss)")
print("- Larger latent dim -> higher KL divergence (the model uses more latent capacity)")
print("- latent_dim=2 is good for visualization but reconstruction is blurry")
print("- Trade-off: more capacity vs more structured/regularized latent space")

---

## 12. Common Mistakes & Debugging Tips

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Forgetting the reparameterization trick | Gradients do not flow to encoder | Use $z = \mu + \sigma \cdot \epsilon$ (never sample directly) |
| Using MSE loss when output has no sigmoid | Reconstruction loss can be negative or very large | Match loss to output activation: BCE with sigmoid, MSE without |
| KL loss too large early in training | Posterior collapse: latent codes ignored, decoder learns marginal | Use KL annealing (gradually increase KL weight from 0 to 1) |
| Not normalizing input to [0, 1] | BCE loss produces NaN | Normalize inputs before training |
| Latent dim too small | Very blurry reconstructions | Increase latent dim (but lose easy visualization) |
| Latent dim too large | VAE behaves like regular autoencoder; unstructured latent space | Decrease latent dim or increase KL weight |
| Confusing $\log \sigma^2$ and $\log \sigma$ | KL divergence formula is wrong by factor of 2 | The standard convention is $\log \sigma^2$ (log-variance) |
| Summing BCE over batch instead of meaning | Loss scale changes with batch size | Use `reduction='sum'` then divide by batch size, or use `reduction='mean'` and scale KL accordingly |

### Debugging Checklist

- **Reconstruction is all gray/uniform**: KL weight too high, posterior collapse, or learning rate too high
- **Generated samples look like noise**: Model underfit, train longer or increase capacity
- **Latent space has no structure**: KL weight too low or latent dim too large
- **NaN in loss**: Check for log of zero or negative values; ensure sigmoid output is clamped

---

**Next notebook:** [02 - GANs Intuition (optional)](02_GANs_Intuition_optional.ipynb)